#### Reading the development and evaluation datasets 

In [3]:
import pandas as pd

dataset = pd.read_csv("C:\\Users\\alias\\Documents\\Polito\\Data Science Lab\\Project\\DSL_Winter_Project_2025\\development.csv")
eval_set = pd.read_csv("C:\\Users\\alias\\Documents\\Polito\\Data Science Lab\\Project\\DSL_Winter_Project_2025\\evaluation.csv")


#### Droping unrelated attributes from dvevelopment dataset

In [4]:
dataset = dataset.drop(columns=["Id", "sampling_rate", "path"])
Id = eval_set["Id"]

#### Droping unrelated attributes from evaluation dataset

In [5]:
eval_set = eval_set.drop(columns=["Id", "sampling_rate", "path"])

#### Correcting data types and misspelled attribute values in both datasets

In [6]:
dataset["tempo"]=dataset["tempo"].str.strip("[]").astype(float)
eval_set["tempo"]=eval_set["tempo"].str.strip("[]").astype(float)
eval_set['gender'] = eval_set['gender'].replace('female', 'famale')

#### separation of X and y

In [7]:
y= dataset["age"]

In [8]:
X = dataset.drop(columns=["age"])

#### train_test_split (80% training set, 20% test set)

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Resetting index of  X_train and X_test becasue train_test_split assigns points randomly to each sets

In [10]:
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

#### Standardizing (z-score) the numerical features. Standardizing the X_train with fit_transform function

In [11]:
from sklearn.preprocessing import StandardScaler

zscore = StandardScaler()

columns_to_normalize = ["mean_pitch", "max_pitch", "min_pitch", "jitter"
                        , "shimmer", "energy", "zcr_mean",
                        "spectral_centroid_mean", "tempo", "hnr",
                        "num_words", "num_characters", "num_pauses",
                        "silence_duration"]

X_train[columns_to_normalize] = zscore.fit_transform(X_train[columns_to_normalize])

#### Standardizing the same features in X_test and evaluation set only using transform  

In [12]:
X_test[columns_to_normalize] = zscore.transform(X_test[columns_to_normalize])
eval_set[columns_to_normalize] = zscore.transform(eval_set[columns_to_normalize])

#### Transforming "gender" to numerical feature with one hot encoding in X_train with fit_transform function

In [13]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output= False, dtype=int, handle_unknown="ignore")
genders = ohe.fit_transform(X_train[["gender"]])
encoded_gender = pd.DataFrame(genders, columns = ohe.get_feature_names_out(["gender"]))

#### Concating the encode feature to X_train

In [14]:
X_train_encoded_gender = pd.concat([X_train, encoded_gender], axis=1)

#### Transforming "gender" to numerical feature with one hot encoding in X_test and evauation set with transform function

In [15]:
genders_test = ohe.transform(X_test[["gender"]])
encoded_gender_test = pd.DataFrame(genders_test, columns = ohe.get_feature_names_out(["gender"]))

genders_eval = ohe.transform(eval_set[["gender"]])
encoded_gender_eval = pd.DataFrame(genders_eval, columns = ohe.get_feature_names_out(["gender"]))

#### Concating the encode feature to X_test and evalution set

In [16]:
X_test_encoded_gender = pd.concat([X_test, encoded_gender_test], axis=1)
eval_encoded_gender = pd.concat([eval_set, encoded_gender_eval], axis=1)

#### Doing the same process, transforming "ethnicity" with one hot encoding 

In [17]:
eths = ohe.fit_transform(X_train_encoded_gender[["ethnicity"]])
encoded_eth = pd.DataFrame(eths, columns = ohe.get_feature_names_out(["ethnicity"]))

In [18]:
X_train_encoded_eth = pd.concat([X_train_encoded_gender, encoded_eth], axis=1)

In [19]:
eths_test = ohe.transform(X_test[["ethnicity"]])
encoded_eth_test = pd.DataFrame(eths_test, columns = ohe.get_feature_names_out(["ethnicity"]))

eths_eval = ohe.transform(eval_set[["ethnicity"]])
encoded_eth_eval = pd.DataFrame(eths_eval, columns = ohe.get_feature_names_out(["ethnicity"]))

In [20]:
X_test_encoded_eth = pd.concat([X_test_encoded_gender, encoded_eth_test], axis=1)
eval_encoded_eth = pd.concat([eval_encoded_gender, encoded_eth_eval], axis=1)

#### Droping the orginal featues "gender" and "ethnicity" after encoding from training, test, and evaluation sets. (Only the transformed version is required for training the model) Now we have the final version of our datasets to apply models on.

In [21]:
X_train_final = X_train_encoded_eth.drop(columns=["gender", "ethnicity"])
X_test_final = X_test_encoded_eth.drop(columns=["gender", "ethnicity"])
eval_final = eval_encoded_eth.drop(columns=["gender", "ethnicity"])